# Model Arena · Slow System en ADA

Entrenar → seleccionar → ajuste final → pronosticar → construir Engine 2/3 → publicar → observar.

Este notebook usa `MacProject`, el adaptador de migración que conserva el TFT, las transformaciones y el monitor contable existentes. El núcleo genérico también expone `ModelArena.fit/predict/tune`; su ejemplo ejecutable está en el notebook 20. La migración del perfil contable al núcleo nuevo no se declara terminada por envolver scripts.

Los archivos de `private/` pertenecen al despliegue y no se suben a GitHub/PyPI. El calendario autoritativo y la configuración están incluidos únicamente en el paquete privado de ADA.

### Protocolo 0.1.7: objetivos independientes
Con family_objective cada familia tiene un estudio para saldo y otro para variación. Cada candidato entrena un objetivo y el ganador se asigna por cuenta y objetivo.
El bundle incluye cinco familias y 25 trials por familia y objetivo. Son hasta 250 intentos y 750 ajustes F1–F3 antes de la poda. El ajuste final conserva 75 épocas por configuración seleccionada.

In [ ]:
from pathlib import Path
import subprocess, sys
from importlib.metadata import version, PackageNotFoundError

VERSION = "0.1.7"
# La instalación conserva los índices y credenciales que ADA ya tenga configurados.
# Si el índice corporativo aún no replica la versión, usa el wheel adjunto.
WHEEL = next(iter(Path.cwd().glob("dist/model_arena_sdk-0.1.7-py3-none-any.whl")), None)
try:
    installed = version("model-arena-sdk")
except PackageNotFoundError:
    installed = None
if installed != VERSION:
    requirement = str(WHEEL.resolve()) if WHEEL else f"model-arena-sdk=={VERSION}"
    command = [sys.executable, "-m", "pip", "install", requirement]
    if WHEEL:
        command.extend(["--find-links", str(WHEEL.parent.resolve())])
    subprocess.run(command, check=True)
from model_arena import __version__
assert __version__ == VERSION, "Reinicia el kernel: quedó importada una versión anterior"
print("Model Arena", __version__, "· Python", sys.version.split()[0])

## Crear o continuar un proyecto
Una corrida nueva genera el calendario con Calendar Studio y private/festivos.csv. Revisa y mantén ese insumo para el período que utilices; el generador no obtiene festivos oficiales. La vista MAC conserva 243 variables. Cada corrida congela datos, reglas, versión y hashes.
Al indicar PROJECT_PATH se reutiliza el calendario congelado de esa corrida, sin regenerarlo.

In [ ]:
from datetime import datetime, timezone
from model_arena.profiles.mac import MacProject
import torch

DEVICE = "cuda"
PROJECT_PATH = ""  # Ruta de un proyecto creado aquí para continuar; vacío crea uno nuevo.
base = Path.cwd().resolve()
if DEVICE == "cuda" and not torch.cuda.is_available():
    raise RuntimeError("Seleccionaste CUDA pero el kernel no tiene PyTorch con GPU. Configura el runtime aprobado de ADA.")
if PROJECT_PATH:
    project = MacProject(PROJECT_PATH)
    project.verify()
else:
    suffix = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    from calendar_studio import CalendarStudio, load_holidays
    calendar = CalendarStudio.from_profile("mac_legacy_v1").generate(
        start="2022-01-01", end="2027-12-31",
        holidays=load_holidays(base / "private/festivos.csv", date_col="fecha"),
    ).view("mac_legacy_v1")
    calendar.validate().raise_for_errors()
    project = MacProject.create(
        base / "runs" / ("mac_" + suffix),
        training_config=base / "private/training.json",
        publication_config=base / "private/publication.json",
        calendar=calendar,
        device=DEVICE,
    )
print("PROJECT_PATH =", repr(str(project.path)))
print(project.status())
plan = project.plan()
print("Plan HPO:", {k: v for k, v in plan.items() if k != "studies"})
info=project.calendar_provenance()
print("Calendario congelado:", {k:info.get(k) for k in ["generator","generator_version","profile","view","start","end","rows","feature_count","calendar_sha256"]})


## 2. Preparación del universo

Ingesta Engine 1 desde Athena o desde el parquet explícito de configuración. `requested_accounts=0` y `account_limit=0`: se usan todas las entrenables elegibles. Las atípicas se conservan para zero-shot. La segmentación por entropía mantiene la fórmula del perfil existente.

In [ ]:
project.prepare()
import json
print(json.loads((project.path / "run/receipts/ingesta.json").read_text())["status"])

## 3. HPO por familia y objetivo

TiDE y NHITS reservan cuatro trials por input X0–X3 y permiten elección libre de input por TPE en los nueve restantes. NLinear, DLinear y NBEATS usan X0. Selecciona familias y presupuesto en private/training.json antes de crear el proyecto.

HPO: máximo 15 épocas por fold, paciencia 4 en validación fija, primera referencia completa por familia/input (y por encoder en Encoder–Decoder). Las losses se conservan por objetivo y cuenta/budget. Sólo trials completos participan en selección. F1–F3 son selección, no Test independiente. Ajuste final: 75 épocas sin poda.


In [ ]:
project.tune()
import polars as pl
curvas = project.path / "run/curvas"
candidates = json.loads((curvas / "candidatos.json").read_text())
print("Candidatos completos:", len(candidates))
audit = pl.read_parquet(curvas / "hpo_trials.parquet")
print(audit.group_by("input", "state").len().sort("input", "state"))
print("Partes de loss por cuenta:", len(list((curvas / "loss_entities").rglob("*.parquet"))))


## 4. Ganador por serie

Primero se identifican finalistas por input y budget y después se asigna un ganador a cada cuenta. Las atípicas reutilizan evidencia de divisa/budget y sus fallbacks. El candidato asignado es único para la serie; distintos cortes pueden tener pesos diferentes de esa misma configuración.

In [ ]:
project.select()
winners = pl.read_parquet(curvas / "winners_FUTURO.parquet")
print(winners.group_by("cohort").len())
print("Configuraciones únicas que requieren ajuste final:", winners["candidate"].n_unique())
winners.head()

## 5. Ajuste final: 75 épocas

Se entrena desde cero cada configuración global con cuentas asignadas, usando toda la historia disponible. Las cuentas zero-shot no entran al ajuste de pesos. La poda es exclusiva del HPO; el ajuste final completa sus 75 épocas.


In [ ]:
project.fit()
print(project.status())

## 6. Pronóstico con modelos congelados

`predict` exige checkpoints existentes. No realiza entrenamiento. FUTURO tiene veinte fechas operativas y dos objetivos; carece de observados futuros para construir validación.

In [ ]:
forecast_path = project.predict()
print("Pronósticos:", forecast_path)
pl.read_parquet(forecast_path).head()

## 7. Postprocessing y layouts

Se generan las curvas históricas con checkpoints admisibles al corte del mismo candidato ganador. La evidencia sigue siendo de selección. Se construyen los alertamientos y las salidas MAC; las cuentas atípicas se evalúan como entrenables en alertamiento, manteniendo su procedencia zero-shot.

Esta etapa todavía no publica en AWS.

In [ ]:
release = project.postprocess()
manifest = json.loads((release / "publication.json").read_text())
print("Publication ID:", manifest["publication_id"])
for dataset in ["engine2", "engine3", "training_learning", "training_entities"]:
    item = manifest["datasets"].get(dataset, {})
    print(dataset, "filas:", item.get("rows"), "columnas:", len(item.get("columns", [])))

## 8. Publicación explícita

Se utiliza el rol AWS de ADA y los destinos del archivo local. El recibo debe corresponder al manifiesto preparado. Si una etapa falla, se conserva el run para inspeccionar y continuar; no se borran corridas anteriores.

In [ ]:
receipt = project.publish()
assert receipt["status"] == "PUBLISHED"
assert receipt["publication_id"] == manifest["publication_id"]
print("Publicación verificada:", receipt["publication_id"])

## 9. Abrir Observatory

El botón abre el monitor Athena existente. A partir de aquí las consultas no entrenan modelos. Para futuras sesiones usa el notebook Fast sin abrir el entorno de entrenamiento.

In [ ]:
monitor = project.observe()
print("PID del monitor:", monitor.pid)